# 📚 Nanophotonics Course, Bar Ilan University 2026
### **Instructor: Dr. Boris Desiatov**

--- 

## Electromagnetic Mode Solving in Nanophotonic Waveguides

Welcome to the **Nanophotonics & Integrated Photonics** mode-solving tutorial! 

In modern integrated optics, light propagation is guided using sub-wavelength structures known as **waveguides**. Analyzing these waveguides requires solving Maxwell's equations in the transverse cross-section to find their **eigenmodes** (stationary electromagnetic field profiles that propagate without changing shape). 

This notebook provides a comprehensive, hands-on guide for first-time users to compute, analyze, and compare two key waveguide architectures:
1. **The Silicon Rectangular Waveguide:** The classical workhorse of modern silicon photonics, which confines light using **Total Internal Reflection (TIR)**.
2. **The Silicon Slot Waveguide:** A high-performance architecture that exploits electromagnetic boundary conditions to achieve **extreme electric field confinement and enhancement** inside a sub-wavelength, low-index slot.

### 💻 Modeling Platform: Tidy3D by Flexcompute
We utilize the state-of-the-art **Tidy3D Mode Solver** to calculate the modal properties. Tidy3D's mode solver formulation solves a 2D generalized eigenvalue problem derived directly from Maxwell's equations, providing rapid and highly accurate computations of the complex effective index ($n_{\text{eff}}$), group index ($n_g$), mode area ($A_{\text{eff}}$), and spatial profiles of all electric and magnetic field components ($E_x, E_y, E_z, H_x, H_y, H_z$).

In [ ]:
import numpy as np
import tidy3d as td
import tidy3d.web as web
from matplotlib import pyplot
from tidy3d.plugins import waveguide
from tidy3d.plugins.mode.web import run as run_mode_solver

# Media used in the examples
si = td.material_library["cSi"]["Li1993_293K"]
sio2 = td.material_library["SiO2"]["Horiba"]

## 🛠️ Part 1.1: The Silicon-on-Insulator (SOI) Single-Mode Waveguide

### 🔬 Physical & Theoretical Overview
The standard Silicon-on-Insulator (SOI) rectangular waveguide consists of a rectangular **Silicon (Si)** core ($n \approx 3.48$ at $\lambda_0 = 1.55\ \mu\text{m}$) fabricated on top of a thick **Silicon Dioxide ($\text{SiO}_2$)** substrate ($n \approx 1.44$), and covered by a $\text{SiO}_2$ upper cladding. This high index contrast ($n_{\text{core}} \approx 3.48$ vs. $n_{\text{clad}} \approx 1.44$) enables strong optical confinement and extremely small bend radii, enabling dense photonic integration.

### 📐 Geometric Dimensions for Single-Mode Operation
For standard single-mode operation, a narrow core of **$500\text{ nm}$ ($0.5\ \mu\text{m}$)** is used with a thickness of **$220\text{ nm}$ ($0.22\ \mu\text{m}$)**. Under these dimensions, the waveguide supports only the fundamental spatial modes for both horizontal (quasi-TE) and vertical (quasi-TM) polarizations.

We will solve for the first **2 modes** (`num_modes=2` in `ModeSpec`) to analyze:
1. **Fundamental quasi-TE ($\text{TE}_0$):** Zero horizontal nodes, dominant horizontal electric field ($E_y$).
2. **Fundamental quasi-TM ($\text{TM}_0$):** Zero vertical nodes, dominant vertical electric field ($E_z$).

Below, we initialize this geometry using Tidy3D's high-level `waveguide` plugin.

In [ ]:
strip_single = waveguide.RectangularDielectric(
    wavelength=1.55,
    core_width=0.5,  # Standard single-mode width (500 nm)
    core_thickness=0.22,
    core_medium=si,
    clad_medium=sio2,
    mode_spec=td.ModeSpec(num_modes=2, group_index_step=True),
)

# Take a look at the waveguide cross-section
_ = strip_single.plot_structures(x=0)

### 🔍 Visualizing the Single-Mode Waveguide Fields
Let's solve for the optical modes. The solver calculates the electromagnetic fields across the cross-sectional $y-z$ plane. The propagation axis is along the $x$-direction.

We plot the dominant electric field component for each mode:
- **Quasi-TE mode (Mode 0):** We plot the horizontal component $E_y$.
- **Quasi-TM mode (Mode 1):** We plot the vertical component $E_z$.

In [ ]:
print(f"Single-Mode Waveguide Effective indices (n_eff): {strip_single.n_eff.values.flatten()}")
print(f"Single-Mode Waveguide Mode areas (A_eff, µm²): {strip_single.mode_area.values.flatten()}")
print(f"Single-Mode Waveguide Group indices (n_group): {strip_single.n_group.values.flatten()}")

fig, ax = pyplot.subplots(1, 2, figsize=(12, 4.5), tight_layout=True)

# Mode 0: Fundamental TE0 (dominant Ey)
strip_single.plot_field("Ey", mode_index=0, ax=ax[0])
ax[0].set_title(f"Fundamental TE0 (Ey), n_eff = {float(strip_single.n_eff.values.flatten()[0]):.4f}")

# Mode 1: Fundamental TM0 (dominant Ez)
strip_single.plot_field("Ez", mode_index=1, ax=ax[1])
ax[1].set_title(f"Fundamental TM0 (Ez), n_eff = {float(strip_single.n_eff.values.flatten()[1]):.4f}")

## 🚀 Part 1.2: Moving to a Multimode Waveguide (Let's Go Multimode!)

### 🧠 The Physics of Multimode Propagation
What happens if we increase the core size? Let's increase the core width of the waveguide from **$0.5\ \mu\text{m}$ to $1.0\ \mu\text{m}$ ($1000\text{ nm}$)** while keeping the thickness at **$220\text{ nm}$**.

Because the core is twice as wide, the spatial confinement along the horizontal axis is significantly relaxed. As a result, the waveguide becomes **multimode**, meaning it supports higher-order spatial modes (modes with spatial nodes or nulls inside the core).

We will solve for the first **4 modes** (`num_modes=4` in `ModeSpec`) to capture and analyze:
1. **Fundamental quasi-TE ($\text{TE}_0$):** Zero horizontal nodes, dominant $E_y$.
2. **First-Order quasi-TE ($\text{TE}_1$):** One horizontal node, dominant $E_y$.
3. **Fundamental quasi-TM ($\text{TM}_0$):** Zero vertical nodes, dominant $E_z$.
4. **First-Order quasi-TM ($\text{TM}_1$):** One vertical node, dominant $E_z$.

In [ ]:
strip_multi = waveguide.RectangularDielectric(
    wavelength=1.55,
    core_width=1.0,  # Increased core width to 1.0 micron (1000 nm)
    core_thickness=0.22,
    core_medium=si,
    clad_medium=sio2,
    # Solve for 4 modes to capture higher-order spatial profiles
    mode_spec=td.ModeSpec(num_modes=4, group_index_step=True),
)

# Take a look at the waveguide cross-section
_ = strip_multi.plot_structures(x=0)

### 🔍 Visualizing the Multimode Waveguide Fields
We solve for the modes and plot the dominant electric field component for each of the 4 modes. Notice the distinct horizontal and vertical node shapes in the first-order spatial modes!

In [ ]:
print(f"Multimode Waveguide Effective indices (n_eff): {strip_multi.n_eff.values.flatten()}")
print(f"Multimode Waveguide Mode areas (A_eff, µm²): {strip_multi.mode_area.values.flatten()}")
print(f"Multimode Waveguide Group indices (n_group): {strip_multi.n_group.values.flatten()}")

fig, ax = pyplot.subplots(2, 2, figsize=(12, 8), tight_layout=True)

# Mode 0: TE0 (dominant Ey)
strip_multi.plot_field("Ey", mode_index=0, ax=ax[0, 0])
ax[0, 0].set_title(f"Mode 0: TE0 (Ey), n_eff = {float(strip_multi.n_eff.values.flatten()[0]):.4f}")

# Mode 1: TE1 (dominant Ey)
strip_multi.plot_field("Ey", mode_index=1, ax=ax[0, 1])
ax[0, 1].set_title(f"Mode 1: TE1 (Ey), n_eff = {float(strip_multi.n_eff.values.flatten()[1]):.4f}")

# Mode 2: TM0 (dominant Ez)
strip_multi.plot_field("Ez", mode_index=2, ax=ax[1, 0])
ax[1, 0].set_title(f"Mode 2: TM0 (Ez), n_eff = {float(strip_multi.n_eff.values.flatten()[2]):.4f}")

# Mode 3: TM1 (dominant Ez)
strip_multi.plot_field("Ez", mode_index=3, ax=ax[1, 1])
ax[1, 1].set_title(f"Mode 3: TM1 (Ez), n_eff = {float(strip_multi.n_eff.values.flatten()[3]):.4f}")

### 🌟 Visualizing Total Electric Field Intensity ($|\mathbf{E}|^2$)
To capture the absolute energy concentration inside the core, we plot the total **electric field intensity ($|\mathbf{E}|^2$)**, or $E^2$, which represents the spatial distribution of the optical energy density:
$$U_e = \frac{1}{2} \epsilon_0 \epsilon_r |\mathbf{E}|^2$$

Plotting $|\mathbf{E}|^2$ shows the absolute spatial concentration of optical power across the core and cladding, regardless of polarization. We do this by calling `plot_field` with `field_name="E"` and `val="abs^2"`.

In [ ]:
fig, ax = pyplot.subplots(2, 2, figsize=(12, 8), tight_layout=True)

# Mode 0: TE0 |E|²
strip_multi.plot_field("E", val="abs^2", mode_index=0, ax=ax[0, 0])
ax[0, 0].set_title(f"Mode 0: TE0 |E|² Intensity")

# Mode 1: TE1 |E|²
strip_multi.plot_field("E", val="abs^2", mode_index=1, ax=ax[0, 1])
ax[0, 1].set_title(f"Mode 1: TE1 |E|² Intensity")

# Mode 2: TM0 |E|²
strip_multi.plot_field("E", val="abs^2", mode_index=2, ax=ax[1, 0])
ax[1, 0].set_title(f"Mode 2: TM0 |E|² Intensity")

# Mode 3: TM1 |E|²
strip_multi.plot_field("E", val="abs^2", mode_index=3, ax=ax[1, 1])
ax[1, 1].set_title(f"Mode 3: TM1 |E|² Intensity")

### 📊 Physical Parameter Analysis: Single-Mode vs. Multimode

Comparing the single-mode ($0.5\ \mu\text{m}$ width) and multimode ($1.0\ \mu\text{m}$ width) waveguide outputs, notice these key physical trends:

1. **Effective Refractive Index ($n_{\text{eff}}$):** 
   - In the single-mode waveguide, the fundamental TE0 mode has $n_{\text{eff}} \approx 2.48$, while the fundamental TM0 has $n_{\text{eff}} \approx 1.83$.
   - In the multimode waveguide, the fundamental TE0 mode's effective index jumps to $n_{\text{eff}} \approx 2.85$! This occurs because the wider Silicon core holds much more of the mode's electric energy within itself, bringing $n_{\text{eff}}$ closer to Silicon's bulk index ($n_{\text{core}} \approx 3.48$).
   - The higher-order spatial modes show progressively lower $n_{\text{eff}}$ values as their broader spatial fields extend deeper into the Silica cladding.

2. **Group Index ($n_g$):**
   - High structural dispersion causes the group index $n_g$ (which determines signal propagation speed: $v_g = c_0/n_g$) to be significantly higher than the material refractive index of silicon itself.

3. **Effective Mode Area ($A_{\text{eff}}$):**
   - Moving to a wider waveguide allows the fundamental mode to squeeze more tightly inside the core, reducing its spatial mode area, while higher-order modes have much larger effective areas due to spatial spreading and structural null nodes.

In [ ]:
# Simple verification check
print("Multimode waveguide successfully solved for 4 guided modes.")

## ⚡ Part 2: The Silicon Slot Waveguide

### 🧠 The Physics of Slot Confinement (Almeida et al., 2004)
In a conventional strip waveguide, light is confined within the high-index silicon core via Total Internal Reflection. However, if we place two high-index silicon cores (called "rails") extremely close to each other (separated by a sub-wavelength distance, typically $50 - 100\text{ nm}$), a very unique physical phenomenon occurs.

According to classical electrodynamics, the normal component of the **electric displacement field ($\mathbf{D}$)** must be continuous across any dielectric interface:
$$D_{1,n} = D_{2,n} \implies \epsilon_1 E_{1,n} = \epsilon_2 E_{2,n}$$

If we express this boundary condition in terms of the refractive index ($n = \sqrt{\epsilon}$), we get:
$$n_1^2 E_{1,n} = n_2^2 E_{2,n} \implies E_{2,n} = \left( \frac{n_1}{n_2} \right)^2 E_{1,n}$$

If Region 1 is **Silicon** ($n_{\text{core}} \approx 3.48$) and Region 2 is **Silica** ($n_{\text{slot}} \approx 1.44$), then at the vertical interfaces (where the normal direction is $y$):
$$E_{\text{slot}} = \left( \frac{3.48}{1.44} \right)^2 E_{\text{Si}} \approx 5.83 \times E_{\text{Si}}$$

This means the horizontal electric field $E_y$ undergoes a massive **jump of nearly 6 times** in amplitude inside the low-index slot region! Because the slot width is extremely narrow, the fields from both interfaces overlap constructively, resulting in an exceptionally strong optical intensity enhancement and tight spatial confinement inside the **low-index** slot.

This makes slot waveguides outstanding candidates for:
- **Optical Sensing & Biosensing:** Interacting directly with molecules placed inside the slot.
- **Electro-Optic Modulators:** Infiltrating the slot with electro-optic polymers or 2D materials.
- **Nonlinear Optics:** Exploiting high field intensity for efficient frequency conversion.

Let's set up the slot waveguide in Tidy3D! We will construct two parallel Silicon rails separated by a $100\text{ nm}$ slot, surrounded by a $\text{SiO}_2$ background cladding.

In [ ]:
# Geometry parameters
w_rail = 0.22      
h_rail = 0.22      
w_slot = 0.1       
wavelength = 1.55

# Calculate positions
offset = (w_rail + w_slot) / 2

rail_left = td.Structure(
    geometry=td.Box(center=(0, -offset, 0), size=(td.inf, w_rail, h_rail)),
    medium=si
)

rail_right = td.Structure(
    geometry=td.Box(center=(0, offset, 0), size=(td.inf, w_rail, h_rail)),
    medium=si
)

# Simulation domain
sim_size = (0.1, 3.0, 2.0) 
mode_plane = td.Box(center=(0, 0, 0), size=(0, 3.0, 2.0))

# Fixed Simulation call: added wavelength to grid_spec
sim = td.Simulation(
    size=sim_size,
    grid_spec=td.GridSpec.auto(min_steps_per_wvl=40, wavelength=wavelength),
    structures=[rail_left, rail_right],
    medium=sio2, 
    run_time=1e-12,
)

mode_solver = td.plugins.mode.ModeSolver(
    simulation=sim,
    plane=mode_plane,
    mode_spec=td.ModeSpec(num_modes=2, target_neff=2.0),
    freqs=[td.C_0 / wavelength],
)

### 🗺️ Visualizing the Slot Waveguide Cross-Section
Before running the solver, let's verify our geometric construction. We plot the refractive index/structure permittivity cross-section to ensure that the two silicon rails are properly positioned and separated by the $100\text{ nm}$ slot.

In [ ]:
# Take a look at the waveguide cross-section
_ = sim.plot_structures(x=0)

### 🏃‍♂️ Running the Mode Solver
Now we run the eigenvalue solver to calculate the slot waveguide modes. We plot the transverse horizontal electric field ($E_y$). 

Look closely at the interface transitions! You should observe a very clear physical manifestation of the electromagnetic boundary conditions: the electric field spikes dramatically inside the low-index slot region.

In [ ]:
# Run the solver
mode_data = mode_solver.solve()

# Access results
print(f"Effective indices: {mode_data.n_eff.values.flatten()}")

# Plot horizontal field component Ey
mode_solver.plot_field("Ey", mode_index=0)

### 🌟 Plotting the Total Electric Field Magnitude ($|\mathbf{E}|$)
To appreciate the extreme concentration of energy, we plot the total electric field magnitude:
$$|\mathbf{E}| = \sqrt{|E_x|^2 + |E_y|^2 + |E_z|^2}$$

Notice how the optical energy is almost completely localized within the $100\text{ nm}$ central slot. This is a remarkable achievement in nanophotonics: guiding and squeezing light into a region far smaller than the diffraction limit of the light itself!

In [ ]:
mode_solver.plot_field("E", mode_index=0)

## 📝 Key Takeaways & Student Exercises

### 🎓 Core Concepts Learned
1. **Rectangular Waveguides** confine light through **Total Internal Reflection (TIR)**. A standard single-mode design ($500\text{ nm}$ width) supports only fundamental TE0 and TM0 modes. Increasing the width to $1.0\ \mu\text{m}$ causes the waveguide to support higher-order spatial modes (TE1, TM1).
2. **Electric Field Intensity ($E^2$)** represents the absolute spatial distribution of electromagnetic energy, proportional to the energy density $U_e$.
3. **Slot Waveguides** confine light using the **continuity of the normal displacement field ($D_n$)** at high-contrast dielectric interfaces, causing a massive field jump in the narrow slot region.

---

### ✏️ Student Homework & Self-Study Exercises

#### 🔬 Exercise 1: Rectangular Waveguide Width Sweep & Cutoff Analysis
- **Task:** Keep the core thickness $h = 220\text{ nm}$ constant. Sweep the core width $w$ from $300\text{ nm}$ to $1200\text{ nm}$ in steps of $100\text{ nm}$.
- **Questions:** 
  1. At what exact core width does the waveguide transition from single-mode to multimode? 
  2. Plot the effective index $n_{\text{eff}}$ of the first 4 modes as a function of width. Why do the indices increase as the width becomes larger?

#### 🔬 Exercise 2: Polarization Selectivity in Slot Waveguides
- **Task:** Plot the field profile of the second solved mode (quasi-TM mode) in the slot waveguide.
- **Questions:**
  1. Why is there no field enhancement in the slot for the quasi-TM mode? 
  2. *Hint:* Think about which electric field component ($E_y$ vs. $E_z$) is normal to the vertical rail interfaces, and write down the relevant boundary condition.